# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. The dataset is defined by a Croissant schema and provides AI-ready mental health survey data including demographic details and widely used mental health indicators.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name} (Published: {metadata.datePublished})\n\n{metadata.description}\n\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Let's review the available record sets in the Croissant schema. We will also enumerate the available fields/columns and their `@id` references for each record set. All operations reference dataset components by their `@id` for consistency and reproducibility.

In [ ]:
# List record sets and their fields by @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - Record Set Name: {rs.name}")
    print(f"    @id: {rs.id}")
    print("    Fields (@id):")
    for field in rs.fields:
        print(f"     - {field.name} (@id: {field.id}) Data type: {field.data_type}")
    print()
if not record_sets:
    print("No record sets found in schema.")

## 3. Data Extraction
Below, we will programmatically extract data from each record set into a dictionary of `DataFrame`s (using record set `@id` for indexing). The columns in each DataFrame correspond to field `@id`s. This ensures all data manipulations remain referential and unambiguous.

In [ ]:
dataframes = {}
rs_ids = [rs.id for rs in dataset.record_sets]
print("Extracting records for each record set:\n")
for rs in dataset.record_sets:
    print(f"Loading: {rs.name} (@id: {rs.id})")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Number of records: {len(df)}\n  Fields: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"  Failed to extract records: {e}\n")

# For demonstration, pick the FIRST available record set for subsequent analysis:
if len(rs_ids) > 0:
    main_record_set_id = rs_ids[0]
    print(f"Main record set for analysis: {main_record_set_id}")
else:
    raise ValueError("No available record set in the dataset.")
print("\nAvailable columns in the main record set:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Select a numeric field and perform basic filtering, normalization, and grouping. Ensure field selection and operations reference fields by their `@id` values.

In [ ]:
main_df = dataframes[main_record_set_id]

# Select a numeric field for EDA: auto-select the first float/int field
numeric_field_id = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            if field.data_type and ("Float" in field.data_type or "Integer" in field.data_type or "Number" in field.data_type):
                numeric_field_id = field.id
                print(f"Using numeric field: {field.name} (@id: {numeric_field_id}) for EDA")
                break
        break
if not numeric_field_id:
    print("No numeric field found.")

# Pick a threshold for demonstration if data exists
if numeric_field_id and numeric_field_id in main_df.columns:
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Number of records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by a categorical/group field (first non-numeric field)
    group_field_id = None
    for rs in dataset.record_sets:
        if rs.id == main_record_set_id:
            for field in rs.fields:
                if field.id != numeric_field_id and (field.data_type and ("Text" in field.data_type or "String" in str(field.data_type))) and field.id in main_df.columns:
                    group_field_id = field.id
                    print(f"Grouping by: {field.name} (@id: {group_field_id})")
                    break
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization

Here, we visualize the distribution of the selected numeric field and grouped means (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field_id and numeric_field_id in main_df.columns and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    fig, ax = plt.subplots(1,2, figsize=(14,5))
    # Distribution of all values
    sns.histplot(main_df[numeric_field_id], bins=20, ax=ax[0], kde=True)
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)

    # Distribution of normalized if computed
    if f"{numeric_field_id}_normalized" in filtered_df:
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, ax=ax[1], kde=True)
        ax[1].set_title(f"Distribution of {numeric_field_id} (Normalized) in Filtered Records")
        ax[1].set_xlabel(f"{numeric_field_id}_normalized")
    plt.tight_layout()
    plt.show()
    # If grouped values exist, plot
    if 'grouped_df' in locals() and group_field_id and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we loaded the FAIR² Mental Health Survey Dataset using `mlcroissant`, enumerated its record sets, inspected fields and records by their `@id`, and conducted a brief EDA including filtering, normalization, grouping, and visualization. The dataset's use of Croissant ensures all data manipulations remain transparent and reproducible via persistent entity identifiers.

**Key Findings:**
- Successfully loaded all record sets and extracted DataFrames using `mlcroissant`.
- Demonstrated EDA on a selected numeric field (e.g., PHQ-9 or GAD-7 score).
- Visualized data distribution and grouped statistics by a categorical attribute.

You may continue with further domain-specific analyses, advanced modeling, or export processed data for downstream applications.
